# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# (Colab) Lấy repo về và chuyển vào thư mục repo. Chạy local trong repo thì cell này không làm gì.
import os, sys
REPO_URL = "https://github.com/2003congthinh/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if "google.colab" in sys.modules and not os.path.exists("scripts/score.py"):
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# Google Drive: sao lưu adapter, dự đoán và bằng chứng (session Colab miễn phí có thể bị ngắt giữa chừng).
# Session mới: restore() chép lại work/ và report/ từ Drive về repo để chạy tiếp, không phải huấn luyện lại.
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_DIR = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/day10_vlm")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)


def backup(path):
    """Chép file/thư mục `path` (đường dẫn tương đối trong repo) sang Drive."""
    if DRIVE_DIR is None:
        return
    path = Path(path)
    dst = DRIVE_DIR / path
    if path.is_dir():
        shutil.copytree(path, dst, dirs_exist_ok=True)
    elif path.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dst)


def restore():
    """Chép work/ và report/ từ Drive về repo (không ghi đè file đang có)."""
    if DRIVE_DIR is None:
        return
    n = 0
    for sub in ("work", "report"):
        src = DRIVE_DIR / sub
        for f in (src.rglob("*") if src.exists() else []):
            dst = Path(sub) / f.relative_to(src)
            if f.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
                n += 1
    print(f"restore: chép {n} file từ {DRIVE_DIR}")


restore()

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh

import json, math, random, re, subprocess, time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

from scripts.labcommon import CLASSES, read_jsonl, write_jsonl

DATA = Path("data")
SPLITS_DIR = DATA / "splits"
WORK = Path("work")
EVID = Path("report/evidence")
FIG = Path("report/figures")
for d in (WORK, EVID, FIG):
    d.mkdir(parents=True, exist_ok=True)
COLORS = {"pothole": "#e6194b", "crack": "#f58231", "manhole": "#4363d8"}

splits = {s: read_jsonl(SPLITS_DIR / f"{s}.jsonl") for s in ("train", "val", "test")}
META = json.loads((SPLITS_DIR / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", META["dataset_version"])
for s, rows in splits.items():
    c = Counter(o["label"] for r in rows for o in r["objects"])
    print(f"  {s:5s} {len(rows):5d} ảnh  {sum(c.values()):5d} box  "
          + "  ".join(f"{k}={c[k]}" for k in CLASSES))

# Không session nào được nằm ở hai split
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not sessions[a] & sessions[b], f"session trùng giữa {a} và {b}: {sessions[a] & sessions[b]}"
print("số session:", {s: len(v) for s, v in sessions.items()}, "→ không session nào nằm ở hai split")

# Nhãn lỗi: box có w = 0 hoặc h = 0 (prepare_data.py chỉ kiểm tra giá trị nằm trong [0,1])
bad = [(s, r["image_id"], o) for s, rows in splits.items() for r in rows for o in r["objects"]
       if o["yolo"][2] <= 0 or o["yolo"][3] <= 0]
print("box suy biến (w=0 hoặc h=0):", bad)


def yolo_to_pixels(yolo, width, height):
    """YOLO (xc, yc, w, h) chuẩn hoá [0,1] → (x1, y1, x2, y2) pixel (số thực)."""
    xc, yc, w, h = yolo
    return ((xc - w / 2) * width, (yc - h / 2) * height, (xc + w / 2) * width, (yc + h / 2) * height)


def load_image(entry):
    return Image.open(DATA / entry["image_path"]).convert("RGB")


def draw_boxes(ax, boxes, color=None, ls="-", lw=1.8, show_label=True):
    """Vẽ list[(label, (x1, y1, x2, y2) pixel)] lên một trục matplotlib đã imshow ảnh."""
    for lab, (x1, y1, x2, y2) in boxes:
        c = color or COLORS.get(lab, "lime")
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=c, lw=lw, ls=ls))
        if show_label:
            ax.text(x1, max(y1 - 3, 8), lab, color="white", fontsize=6,
                    bbox=dict(fc=c, ec="none", pad=0.8))


# 6 ảnh train, mỗi ảnh lấy từ một session khác nhau
rng = random.Random(0)
by_session = defaultdict(list)
for r in splits["train"]:
    by_session[r["session"]].append(r)
samples = [rng.choice(by_session[s]) for s in rng.sample(sorted(by_session), 6)]
fig, axes = plt.subplots(2, 3, figsize=(15, 6.2))
for ax, e in zip(axes.flat, samples):
    ax.imshow(load_image(e))
    ax.axis("off")
    draw_boxes(ax, [(o["label"], yolo_to_pixels(o["yolo"], e["width"], e["height"])) for o in e["objects"]])
    ax.set_title(f"{e['image_id']}  [{e['session']}]  {len(e['objects'])} box", fontsize=8)
fig.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in COLORS.items()], loc="lower center", ncol=3)
fig.suptitle("6 ảnh train (6 phiên quay khác nhau) với box gốc từ nhãn YOLO")
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIG / "data_samples.png", dpi=110)
plt.show()

# Phân bố lớp theo split và kích thước box
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.2))
x, wbar = np.arange(len(CLASSES)), 0.27
for i, (s, rows) in enumerate(splits.items()):
    c = Counter(o["label"] for r in rows for o in r["objects"])
    bars = a1.bar(x + (i - 1) * wbar, [c[k] for k in CLASSES], wbar, label=f"{s} ({len(rows)} ảnh)")
    a1.bar_label(bars, fontsize=7)
a1.set_xticks(x, CLASSES)
a1.set_ylabel("số box")
a1.set_title("Số box từng lớp theo split (chia theo phiên quay)")
a1.legend()
for k in CLASSES:
    areas = np.array([o["yolo"][2] * o["yolo"][3] * 100 for rows in splits.values() for r in rows
                      for o in r["objects"] if o["label"] == k])
    a2.hist(np.log10(np.clip(areas, 1e-3, None)), bins=40, alpha=0.55, color=COLORS[k],
            label=f"{k}: trung vị {np.median(areas):.2f}% khung hình")
a2.set_xlabel("log10(diện tích box, % khung hình)")
a2.set_ylabel("số box")
a2.set_title("Box rất nhỏ: phần lớn < 1% khung hình")
a2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "data_stats.png", dpi=110)
plt.show()

In [ ]:
# Câu 6: đo rò rỉ nếu chia NGẪU NHIÊN. Mỗi ảnh test tìm ảnh train gần nhất (thumbnail xám 32×18, khoảng cách L2).
# So sánh split theo phiên (của đề) với một split ngẫu nhiên cùng tỉ lệ ảnh test.
def thumbs(entries):
    return np.stack([np.asarray(load_image(e).convert("L").resize((32, 18)), dtype=np.float32).ravel() / 255
                     for e in entries])


def nn_dist(test_x, train_x):
    d = np.sqrt(np.clip((test_x ** 2).sum(1)[:, None] + (train_x ** 2).sum(1)[None, :] - 2 * test_x @ train_x.T, 0, None))
    return d.min(1), d.argmin(1)


def leak_check(seed=0):
    every = splits["train"] + splits["val"] + splits["test"]
    X = thumbs(every)
    n_tr, n_te = len(splits["train"]), len(splits["test"])
    idx_tr, idx_te = np.arange(n_tr), np.arange(len(every) - n_te, len(every))
    d_sess, nn_sess = nn_dist(X[idx_te], X[idx_tr])
    perm = np.random.default_rng(seed).permutation(len(every))
    r_te, r_tr = perm[:n_te], perm[n_te:n_te + n_tr]
    d_rand, nn_rand = nn_dist(X[r_te], X[r_tr])
    thr = np.percentile(d_sess, 5)        # "gần trùng" = gần hơn 95% cặp test–train của split theo phiên
    print(f"khoảng cách tới ảnh train gần nhất (trung vị): theo phiên {np.median(d_sess):.2f} | "
          f"ngẫu nhiên {np.median(d_rand):.2f}")
    print(f"tỉ lệ ảnh test có 'bản gần trùng' trong train (d < {thr:.2f}): theo phiên {np.mean(d_sess < thr):.1%} | "
          f"ngẫu nhiên {np.mean(d_rand < thr):.1%}")
    i = int(np.argmin(d_rand))
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))
    j = int(np.argmin(np.abs(d_sess - np.median(d_sess))))
    pairs = [(every[r_te[i]], f"test (chia ngẫu nhiên)"), (every[r_tr[nn_rand[i]]], f"train gần nhất: d={d_rand[i]:.2f}"),
             (every[idx_te[j]], "test (chia theo phiên)"), (every[idx_tr[nn_sess[j]]], f"train gần nhất: d={d_sess[j]:.2f}")]
    for ax, (e, t) in zip(axes, pairs):
        ax.imshow(load_image(e))
        ax.axis("off")
        ax.set_title(f"{t}\n{e['image_id']}", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / "leak_check.png", dpi=110)
    plt.show()
    return {"median_session": float(np.median(d_sess)), "median_random": float(np.median(d_rand)),
            "thr": float(thr), "near_dup_session": float(np.mean(d_sess < thr)),
            "near_dup_random": float(np.mean(d_rand < thr))}


LEAK = leak_check()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

IMG_W, IMG_H = 640, 360


def coord_scale(coord, width=IMG_W, height=IMG_H):
    """Hệ số (sx, sy) để đổi toạ độ chuẩn hoá [0,1] sang thang đầu ra ("norm1000" hoặc "pixel")."""
    return (1000, 1000) if coord == "norm1000" else (width, height)


def yolo_to_box(label, yolo, coord="norm1000", width=IMG_W, height=IMG_H):
    xc, yc, w, h = yolo
    sx, sy = coord_scale(coord, width, height)
    x1, y1 = round((xc - w / 2) * sx), round((yc - h / 2) * sy)
    x2, y2 = round((xc + w / 2) * sx), round((yc + h / 2) * sy)
    x1, x2 = min(max(x1, 0), sx), min(max(x2, 0), sx)
    y1, y2 = min(max(y1, 0), sy), min(max(y2, 0), sy)
    # Box rất mảnh có thể bị làm tròn thành x2 <= x1: nới 1 đơn vị, vì score.parse_output
    # coi CẢ output là hỏng nếu có một box suy biến — model không được học điều đó.
    if x2 <= x1:
        x1, x2 = (x1, x1 + 1) if x1 < sx else (x1 - 1, x1)
    if y2 <= y1:
        y1, y2 = (y1, y1 + 1) if y1 < sy else (y1 - 1, y1)
    return {"label": label, "bbox_2d": [int(x1), int(y1), int(x2), int(y2)]}


def objects_to_answer(objects, coord="norm1000", width=IMG_W, height=IMG_H):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord, width, height)
             for o in objects if o["yolo"][2] > 0 and o["yolo"][3] > 0]   # bỏ nhãn lỗi w=0/h=0
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))       # trên→dưới, trái→phải
    return json.dumps(boxes, separators=(",", ":"))


def box_to_pixels(bbox, coord="norm1000", width=IMG_W, height=IMG_H):
    """Ngược lại của yolo_to_box: bbox_2d theo thang `coord` → (x1, y1, x2, y2) pixel."""
    sx, sy = coord_scale(coord, width, height)
    x1, y1, x2, y2 = bbox
    return (x1 * width / sx, y1 * height / sy, x2 * width / sx, y2 * height / sy)


def make_prompt(coord="norm1000", width=IMG_W, height=IMG_H):
    if coord == "norm1000":
        scale = ("integers on a 0-1000 scale relative to the image size "
                 "(x = 1000 * pixel_x / image_width, y = 1000 * pixel_y / image_height); "
                 "(0,0) is the top-left corner and (1000,1000) is the bottom-right corner")
    else:
        scale = (f"absolute integer pixel coordinates of the {width}x{height} image "
                 f"(x from 0 to {width}, y from 0 to {height}); (0,0) is the top-left corner")
    return ("Locate every road damage in this dashcam image. There are exactly 3 classes: "
            "pothole, crack, manhole.\n"
            "Answer with ONLY a JSON list and nothing else (no markdown, no explanation), e.g. "
            '[{"label": "crack", "bbox_2d": [x1, y1, x2, y2]}]\n'
            f"bbox_2d is the box [x1, y1, x2, y2] in {scale}. "
            "Return [] if there is no damage.")


PROMPT = make_prompt("norm1000")
PROMPT_PIXEL = make_prompt("pixel")
print(PROMPT, "\n")

# Kiểm tra: YOLO → JSON 0–1000 → pixel phải trùng box gốc (đã kẹp vào khung ảnh); sai số chỉ do làm tròn số nguyên
def clamp_px(b):
    return (min(max(b[0], 0), IMG_W), min(max(b[1], 0), IMG_H), min(max(b[2], 0), IMG_W), min(max(b[3], 0), IMG_H))


all_objs = [o for rows in splits.values() for r in rows for o in r["objects"] if o["yolo"][2] > 0 and o["yolo"][3] > 0]
n_out = sum(clamp_px(yolo_to_pixels(o["yolo"], IMG_W, IMG_H)) != yolo_to_pixels(o["yolo"], IMG_W, IMG_H) for o in all_objs)
print(f"{n_out}/{len(all_objs)} box YOLO tràn ra ngoài khung ảnh → bị kẹp về [0, 1000]")
for coord in ("norm1000", "pixel"):
    err = max(max(abs(u - v) for u, v in zip(box_to_pixels(yolo_to_box(o["label"], o["yolo"], coord)["bbox_2d"], coord),
                                             clamp_px(yolo_to_pixels(o["yolo"], IMG_W, IMG_H))))
              for o in all_objs)
    print(f"[{coord}] sai số lớn nhất khi quy ngược về pixel ({len(all_objs)} box): {err:.3f} px")
    assert err < 1.0

ex = max(splits["train"], key=lambda r: len(r["objects"]))      # ảnh nhiều box nhất
ans = objects_to_answer(ex["objects"])
print("\nảnh", ex["image_id"], "→", ans)
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.imshow(load_image(ex))
ax.axis("off")
draw_boxes(ax, [(o["label"], yolo_to_pixels(o["yolo"], IMG_W, IMG_H)) for o in ex["objects"]],
           color="lime", lw=3.5, show_label=False)
draw_boxes(ax, [(b["label"], box_to_pixels(b["bbox_2d"])) for b in json.loads(ans)], ls="--", lw=1.5)
ax.set_title("Kiểm tra round-trip: xanh lá = box YOLO gốc, nét đứt = JSON 0–1000 quy ngược về pixel", fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "label_roundtrip.png", dpi=110)
plt.show()

In [ ]:
# Câu 3 (thang toạ độ): "oracle" = đáp án ĐÚNG HOÀN HẢO, chấm với --coord khớp và KHÔNG khớp.
# Không cần GPU: cho thấy riêng việc lệch thang đã làm F1 sụp thế nào.
def coord_mismatch_demo(split="test"):
    rows = splits[split]
    out = {}
    for label_coord in ("norm1000", "pixel"):
        path = WORK / f"oracle_{label_coord}.jsonl"
        write_jsonl(path, [{"image_id": r["image_id"], "raw": objects_to_answer(r["objects"], label_coord)}
                           for r in rows])
        for score_coord in ("norm1000", "pixel"):
            res = subprocess.run([sys.executable, "scripts/score.py", "--pred", str(path), "--split", split,
                                  "--coord", score_coord, "--name", f"oracle-{label_coord}-as-{score_coord}",
                                  "--out", str(WORK / f"oracle_{label_coord}_as_{score_coord}.json"),
                                  "--n-boot", "200"], capture_output=True, text=True, check=True)
            d = json.loads((WORK / f"oracle_{label_coord}_as_{score_coord}.json").read_text(encoding="utf-8"))
            out[(label_coord, score_coord)] = d
            print(f"nhãn {label_coord:8s} chấm --coord {score_coord:8s}: F1={d['micro']['f1']:.3f} "
                  f"P={d['micro']['precision']:.3f} R={d['micro']['recall']:.3f} meanIoU={d['mean_iou']:.3f}")
    return out


ORACLE = coord_mismatch_demo()

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9